# BootstrapFewShot

Bootstraps successful reasoning traces, then attaches them as demonstrations.

This notebook is self-contained and targets the stable DSPy 3.x API pinned by
`requirements.txt`. Its default data and optimizer budgets are smoke-test sized;
set `TRAIN_LIMIT=0`, `VAL_LIMIT=0`, and `EVAL_LIMIT=0` for the complete split.

In [ ]:
from pathlib import Path

_requirements = next(
    (
        candidate / "requirements.txt"
        for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (candidate / "requirements.txt").is_file()
    ),
    None,
)
if _requirements is None:
    raise FileNotFoundError("Could not locate requirements.txt from the current working directory.")
%pip install -r "{_requirements}" -q 

In [ ]:
import os
import random
from pathlib import Path

import dspy
import pandas as pd
from dotenv import load_dotenv

# Resolve paths from the repo root, chapter06/, or any nested repo directory.
cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in (cwd, *cwd.parents)
        if (candidate / "requirements.txt").is_file()
        and (candidate / "data" / "ai_vs_human200.csv").is_file()
    ),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        f"Could not locate the repository root from {cwd}. "
        "Start Jupyter somewhere inside the cloned repository."
    )
DATA_PATH = REPO_ROOT / "data" / "ai_vs_human200.csv"

load_dotenv(REPO_ROOT / ".env")
if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError("Set OPENAI_API_KEY in the repo's .env file before running this notebook.")

# Chapter 6 defaults: Luna for high-volume task calls, Sol for reflection.
# Override either slug without editing the notebook.
TASK_MODEL = os.getenv("TASK_MODEL", "openai/gpt-5.6-luna")
REFLECTION_MODEL = os.getenv("REFLECTION_MODEL", "openai/gpt-5.6-sol")
NUM_THREADS = int(os.getenv("DSPY_NUM_THREADS", "4"))
TRAIN_LIMIT = int(os.getenv("TRAIN_LIMIT", "32"))
VAL_LIMIT = int(os.getenv("VAL_LIMIT", "12"))
EVAL_LIMIT = int(os.getenv("EVAL_LIMIT", "12"))

# DSPy 3.2 treats GPT-5-family models as reasoning models and rejects small
# max_tokens caps. Leave the provider default in place rather than setting an
# invalid value below 16,000.
task_lm = dspy.LM(TASK_MODEL)
reflection_lm = dspy.LM(REFLECTION_MODEL)
dspy.configure(lm=task_lm)

print(f"DSPy {dspy.__version__}")
print(f"Task model: {TASK_MODEL}")
print(f"Reflection model: {REFLECTION_MODEL}")

In [ ]:
class DetectAIText(dspy.Signature):
    """Decide whether the supplied passage was generated by an AI."""

    text: str = dspy.InputField(desc="Passage to classify")
    is_ai: bool = dspy.OutputField(desc="True for AI-generated text; otherwise False")


class AIDetector(dspy.Module):
    def __init__(self):
        super().__init__()
        self.detect = dspy.ChainOfThought(DetectAIText)

    def forward(self, text: str):
        return self.detect(text=text)


def as_bool(value) -> bool:
    if isinstance(value, bool):
        return value
    normalized = str(value).strip().lower()
    if normalized in {"true", "1", "yes"}:
        return True
    if normalized in {"false", "0", "no"}:
        return False
    raise ValueError(f"Cannot interpret {value!r} as a boolean")


def exact_match(example, prediction, trace=None) -> float:
    return float(as_bool(prediction.is_ai) == bool(example.is_ai))


def feedback_metric(example, prediction, trace=None, **kwargs):
    score = exact_match(example, prediction)
    expert_note = getattr(example, "notes", "")
    feedback = (
        "The classification is correct. Preserve the reasoning cues that led to it."
        if score
        else f"The classification is incorrect. The expected is_ai value is {bool(example.is_ai)}. "
             "Focus on concrete stylistic evidence instead of topic or polish alone."
    )
    if expert_note:
        feedback += f" Expert note: {expert_note}"
    return dspy.Prediction(score=score, feedback=feedback)


frame = pd.read_csv(DATA_PATH)
examples = [
    dspy.Example(
        text=row.text,
        is_ai=as_bool(row.is_ai),
        notes="" if pd.isna(row.notes) else str(row.notes),
    ).with_inputs("text")
    for row in frame.itertuples(index=False)
]
random.Random(42).shuffle(examples)

# Deterministic 50/25/25 split; environment variables keep default runs inexpensive.
train_end = len(examples) // 2
val_end = train_end + len(examples) // 4
full_trainset, full_valset, full_testset = (
    examples[:train_end], examples[train_end:val_end], examples[val_end:]
)
trainset = full_trainset[:TRAIN_LIMIT] if TRAIN_LIMIT else full_trainset
valset = full_valset[:VAL_LIMIT] if VAL_LIMIT else full_valset
testset = full_testset[:EVAL_LIMIT] if EVAL_LIMIT else full_testset

def evaluate(program, dataset=testset):
    runner = dspy.Evaluate(
        devset=dataset,
        metric=exact_match,
        num_threads=NUM_THREADS,
        max_errors=max(1, len(dataset)),
        provide_traceback=True,
        failure_score=0.0,
        display_progress=True,
        display_table=5,
    )
    result = runner(program)
    failed = [index for index, (_, prediction, _) in enumerate(result.results) if not hasattr(prediction, "is_ai")]
    if failed:
        raise RuntimeError(f"Evaluation failed for example indexes {failed}; inspect the tracebacks above.")
    return result.score


detector = AIDetector()
print(f"train={len(trainset)}, validation={len(valset)}, test={len(testset)}")

In [ ]:
baseline_score = evaluate(detector)
print(f"Baseline score: {baseline_score}")

In [ ]:
optimizer = dspy.BootstrapFewShot(
    metric=exact_match,
    max_bootstrapped_demos=2,
    max_labeled_demos=2,
    max_rounds=1,
)
optimized_detector = optimizer.compile(detector, trainset=trainset)

In [ ]:
optimized_score = evaluate(optimized_detector)
print(f"Optimized score: {optimized_score}")
print(f"Delta: {optimized_score - baseline_score:+.2f}")

example = testset[0]
prediction = optimized_detector(**example.inputs())
print({"expected": example.is_ai, "predicted": prediction.is_ai})